FactScore → "Checks facts against evidence."
SelfCheckGPT → "Checks consistency across generations."
BERTScore → "Checks semantic similarity."
HHEM → "Uses a trained model to estimate hallucination probability."

In [1]:
!pip install sentence-transformers

In [2]:
# SELF-CHECK GPT DEMO
# Hallucination Detection using Answer Consistency

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# STEP 1:
# Simulated multiple LLM responses
# (Normally generated using GPT/Groq/OpenAI API)

question = "Who founded Tesla?"

responses = [
    "Tesla was founded by Martin Eberhard and Marc Tarpenning.",
    "Tesla was founded by Elon Musk.",
    "Tesla was founded by Martin Eberhard and Marc Tarpenning in 2003.",
    "Elon Musk founded Tesla Motors."
]

# STEP 2:
# Load embedding model
# Converts text into vectors

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# STEP 3:
# Generate embeddings

embeddings = model.encode(responses)

# STEP 4:
# Compute pairwise similarity

similarity_matrix = cosine_similarity(embeddings)

print("Similarity Matrix:")
print(similarity_matrix)

# STEP 5:
# Calculate average agreement score

scores = []

for i in range(len(responses)):
    for j in range(i + 1, len(responses)):
        scores.append(similarity_matrix[i][j])

avg_similarity = np.mean(scores)

print("\nAverage Similarity:", round(avg_similarity, 3))

# STEP 6:
# Hallucination Risk Estimation

if avg_similarity > 0.85:
    risk = "LOW"
elif avg_similarity > 0.70:
    risk = "MEDIUM"
else:
    risk = "HIGH"

print("\nHallucination Risk:", risk)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Similarity Matrix:
[[1.0000002  0.68010795 0.96377903 0.60445106]
 [0.68010795 1.0000001  0.6488037  0.9399203 ]
 [0.96377903 0.6488037  1.0000001  0.5836423 ]
 [0.60445106 0.9399203  0.5836423  1.        ]]

Average Similarity: 0.737

Hallucination Risk: MEDIUM


In [12]:
# EXAMPLE FOR ALL HALLUCINATION DETECTION METHODS

question = "What is the capital of France?"

context = """
France is a country in Europe.
Its capital city is Paris.
Paris is the largest city in France.
"""

answer = """
The capital of France is Lyon.
Lyon is the largest city in France.
"""

print("QUESTION:")
print(question)

print("\nCONTEXT:")
print(context)

print("\nGENERATED ANSWER:")
print(answer)

QUESTION:
What is the capital of France?

CONTEXT:

France is a country in Europe.
Its capital city is Paris.
Paris is the largest city in France.


GENERATED ANSWER:

The capital of France is Lyon.
Lyon is the largest city in France.



In [13]:
# FACTSCORE
# Supported Claims / Total Claims

claims = [
    ("Capital", "Lyon"),
    ("Largest City", "Lyon")
]

supported = 0

print("Fact Verification:\n")

for claim_type, value in claims:

    if value in context:
        print(f"✅ Supported: {claim_type} = {value}")
        supported += 1

    else:
        print(f"❌ Hallucinated: {claim_type} = {value}")

factscore = supported / len(claims)

print("\nFactScore:", round(factscore, 2))

Fact Verification:

❌ Hallucinated: Capital = Lyon
❌ Hallucinated: Largest City = Lyon

FactScore: 0.0


In [14]:
# SELF-CHECK GPT
# Compare multiple generated answer

# =====================================================
# SELFCHECKGPT
# =====================================================

from collections import Counter

answers = [
    "The capital of France is Paris.",
    "The capital of France is Lyon.",
    "The capital of France is Paris.",
    "The capital of France is Marseille.",
    "The capital of France is Paris."
]

cities = []

for answer in answers:

    city = answer.split("is")[-1].replace(".", "").strip()

    cities.append(city)

print("Generated Answers:\n")

for city in cities:
    print(city)

counter = Counter(cities)

agreement = counter.most_common(1)[0][1] / len(cities)

print("\nAgreement Score:", round(agreement,2))

if agreement >= 0.8:
    print("Low Hallucination Risk")
elif agreement >= 0.6:
    print("Medium Hallucination Risk")
else:
    print("High Hallucination Risk")

Generated Answers:


Lyon

Marseille


Agreement Score: 0.6
Medium Hallucination Risk


In [15]:
!pip install bert-score -q

In [16]:
# BERTSCORE
# Semantic similarity with reference answer

from bert_score import score

reference = "The capital of France is Paris."

candidate = "The capital of France is Lyon."

P, R, F1 = score(
    [candidate],
    [reference],
    lang="en",
    verbose=False
)

print("Precision :", round(P.mean().item(),3))
print("Recall    :", round(R.mean().item(),3))
print("F1 Score  :", round(F1.mean().item(),3))

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Precision : 0.976
Recall    : 0.976
F1 Score  : 0.976


In [17]:
# HHEM (CONCEPTUAL)
# Context + Answer -> Hallucination Probability

hallucination_probability = 0.95

print(
    f"Hallucination Probability: "
    f"{hallucination_probability:.0%}"
)

if hallucination_probability > 0.7:
    print("❌ Hallucination Detected")

elif hallucination_probability > 0.4:
    print("⚠️ Possible Hallucination")

else:
    print("✅ Faithful Answer")

Hallucination Probability: 95%
❌ Hallucination Detected
